# Track 5 — Smart Self (Personal Health / Activity)
### EP0418 · AI for Real-World Data and Automated Decisions · Starter Pack

**Who this suits:** anyone with a smartwatch, athletes, self-improvement curious.
**Data:** 180 days of daily activity — steps, resting heart rate, sleep, active
minutes. A realistic sample ships with this pack; **swap in your own export** and
the notebook still works.
**Macro lesson:** *Pattern detection in personal data + the privacy conversation.*
**Your built-in baseline:** rolling 7-day average.

---
> ### Before you use your own data — read this
> This is the only track where the dataset is **about a person**. If you export
> your own Apple Health or Strava data, you are handling health information about
> yourself, and possibly about people you exercise with.
>
> - **Do not** commit your real export to a public GitHub repo.
> - **Do not** use a classmate's data without asking, every time.
> - **Do** anonymise before you submit: no names, no GPS traces, no addresses.
> - If you would be uncomfortable with your tutor reading a row of it, use the
>   sample data instead. That is a legitimate choice and costs you no marks.
>
> Your CA3 reflection must address this directly (SLO 10.3).

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.


In [ ]:
import os, urllib.request
# Fetch this pack's data from the course repository if it is not here yet.
for _f in ['smart_self_daily.csv']:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track5_smart_self/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os

CSV = "smart_self_daily.csv"
if not os.path.exists(CSV):
    from google.colab import files; files.upload()      # pick the CSV from the pack

df = pd.read_csv(CSV, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

In [ ]:
# --- FIRST LOOK -------------------------------------------------------
print("rows:", len(df), " range:", df.date.min().date(), "->", df.date.max().date())
span = (df.date.max() - df.date.min()).days + 1
print(f"calendar days in that span: {span}   rows present: {len(df)}   MISSING DAYS: {span - len(df)}")
print("\nzero-step days:", (df.steps == 0).sum(), " <-- watch not worn, or genuinely 0 steps?")
num = df.select_dtypes("number")
print("\n", num.describe().T[["min", "mean", "max"]].round(1).to_string())

### Two traps are already visible above

**Zero is not the same as missing.** On days the watch was not worn, this dataset
records `steps = 0`. That is a *sensor* statement ("I saw nothing"), not a *human*
statement ("I walked nothing"). Leave those zeros in and every average you compute
is dragged down. Decide what to do with them and justify it (SLO 4.2).

**Some days are simply absent.** The date index has holes, so `df.steps.shift(7)`
does **not** reliably mean "one week ago" — it means "seven rows back", which may
be nine days. Reindex to a complete daily calendar before doing anything with
lags (SLO 3.3).

In [ ]:
# --- PLOT -------------------------------------------------------------
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax[0].plot(df.date, df.steps, lw=.8, color="tab:blue")
ax[0].plot(df.date, df.steps.rolling(7).mean(), lw=2, color="darkblue", label="7-day mean")
ax[0].set_ylabel("steps"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(df.date, df.resting_hr, lw=.9, color="tab:red")
ax[1].set_ylabel("resting HR"); ax[1].grid(alpha=.3)
ax[2].plot(df.date, df.sleep_hours, lw=.9, color="tab:purple")
ax[2].set_ylabel("sleep (h)"); ax[2].grid(alpha=.3)
ax[0].set_title("180 days of personal activity data")
plt.tight_layout(); plt.show()

In [ ]:
# --- YOUR BASELINE: ROLLING 7-DAY AVERAGE -----------------------------
# Predict tomorrow's steps as the mean of the last 7 days. That's the bar.

d = df.set_index("date").asfreq("D")           # fill the calendar holes first
pred = d.steps.rolling(7).mean().shift(1)
m = pred.notna() & d.steps.notna()
mae  = (d.steps[m] - pred[m]).abs().mean()
rmse = np.sqrt(((d.steps[m] - pred[m]) ** 2).mean())
print(f"ROLLING 7-DAY BASELINE   MAE {mae:,.0f} steps   RMSE {rmse:,.0f} steps")
print(f"(mean daily steps: {d.steps.mean():,.0f})")

# Same baseline, used as an anomaly detector
roll_m, roll_s = d.steps.rolling(14).mean(), d.steps.rolling(14).std()
z = (d.steps - roll_m) / roll_s
print(f"\ndays flagged |z| > 2 : {(z.abs() > 2).sum()}")
print(d.loc[z.abs() > 2, "steps"].to_string())
print("\n>>> Some of those are real events. Some are the watch being in a drawer.")
print(">>> An anomaly detector cannot tell you which. Only you can. (SLO 5.3)")

---
## Three macro questions to investigate

**1. Can you detect a real-life event you already know happened?**
Somewhere in this data is a stretch where activity collapses and resting heart
rate climbs — an illness or injury. Find its start date from the data alone, then
ask whether you could have detected it *on day 2* rather than day 10. That gap is
the whole design problem in health monitoring (SLO 5.3).

**2. What is the cost of a false alarm to a real person?**
A fitness app that warns "your resting HR is elevated" every time you drink
coffee gets uninstalled. One that stays silent through a genuine problem is worse.
Where do you set the threshold, and who bears the cost of each error (SLO 7.1)?

**3. Would you deploy this on someone else?**
Your prototype runs on your own data, where you can sanity-check every number.
Deploying it for a stranger means it fires on data you cannot verify, about a body
you know nothing about. What breaks? What would you have to add before it were
safe (SLO 10.2)?

## Now hand over to your AI assistant

Everything above was written for you. From here you drive. Open ChatGPT, Claude,
Copilot, or Cursor and work the **prompt → generate → verify → refine** loop
(SLO 9.1). A prompt worth copying looks like this:

> I have a pandas DataFrame `df` with columns `date, steps, resting_hr, sleep_hours, active_minutes`.
> The time column is `date`. Reindex to a complete daily calendar, treat zero-step days as missing rather than as zeros, then flag anomalous days using a 14-day rolling z-score on steps and resting heart rate.
> Show me the code and explain what each step does.

**Then verify before you trust it (SLO 9.2).** Every time:

| Check | How |
|---|---|
| Does it run? | Run the cell. Read the error, don't just re-prompt. |
| Right shape? | `df.shape` before and after — did you lose rows you didn't mean to? |
| Right range? | `df.describe()` — any impossible values (negative rainfall, 300 bpm)? |
| Spot-check | Pick 2 rows by hand and confirm the number is what you'd compute yourself. |
| Beat baseline? | Compare against the baseline above. If it loses, say so honestly. |

**Keep a `verification_log.md`** as you go — one line per AI mistake you caught and
how you fixed it. That log is assessed in CA3, and an empty one reads as
"I did not check", not "the AI was perfect".